In [1]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv
import time
load_dotenv()
local_connection = MongoClient("mongodb://localhost:27017")
klg_connection = MongoClient(os.environ.get("ENTITIES_DB"))
protocol_connection = MongoClient(os.environ.get("DAPP_INFO_DB"))
klg_database = klg_connection["knowledge_graph"]
local_database = local_connection["blockchain_etl"]
protocol_database = protocol_connection["SmartContractLabel"]

In [13]:
chain_collection = {
    "0x38": "bnb",
    "0x89": "poly",
    "0x1": "eth",
    "0xa4b1": "arb"
}
timestamps = [i for i in range(1719792000, 1727740800, 86400)]

In [40]:
def round_timestamp(timestamp, round_time=86400):
    timestamp = int(timestamp)
    timestamp_unit_day = timestamp / round_time
    recover_to_unit_second = int(timestamp_unit_day) * round_time
    return recover_to_unit_second
    
def get_price(chain_id, token, timestamp, price_tokens):
    cursor = price_tokens.get(token)
    if not cursor:
        cursor = klg_database["smart_contracts"].find_one({"_id": f"{chain_id}_{token}".lower()})
        price_tokens[token] = cursor
    price = 0
    
    if cursor.get("priceChangeLogs"):
        tmp = time.time()
        for ts, p in cursor.get("priceChangeLogs").items():
            if int(ts) <= tmp and int(ts) >= timestamp:
                price = p
                tmp = int(ts)
    elif cursor.get("price"):
        price = cursor.get("price")
    return price

def get_protocol_information(chain):
    exchange_rate = {}
    protocols = {}
    underlying = {}
    for item in protocol_database["protocols"].find({
        "chainId": chain,
        "$or":[
            {"_id": {"$regex": "aave"}},
            {"_id": {"$regex": "venus"}},
            {"_id": {"$regex": "compound"}}
        ]
    }):
        if "compound-v3" in item.get("_id") or "morpho-compound" in item.get("_id") or "morpho-aave" in item.get("_id"):
            continue 
        if "aave" in item.get("_id"):
            protocols[item.get("address")] = "aave"
        if "compound" in item.get("_id") or "venus" in item.get("_id"):
            for token in item["reservesList"]:
                ctoken = item['reservesList'][token]['cToken']
                underlying[ctoken] = token
                protocols[ctoken] = "compound"
                exchange_rate[ctoken] =  item['reservesList'][token]["exchangeRate"]
    return exchange_rate, protocols, underlying

In [41]:
result = {}
token_price = {}
for chain, prefix in chain_collection.items():
    exchange_rate, protocols, underlyings = get_protocol_information(chain)
    result[chain] = {
        str(i):{
            "wallets": [],
            "events": 0,
            "amount": 0
        } for i in timestamps
    }
    cursor = local_database[f"{prefix}_lq_events"].find({"contract_address": {"$in": list(protocols.keys())}})
    for event in cursor:
        round_time = round_timestamp(event.get("block_timestamp"))
        if round_time not in timestamps:
            continue
        contract_address = event.get("contract_address")
        amount = 0
        if protocols.get(contract_address) == "aave":
            debt_token = event.get("debt_asset")
            amount = event.get("debt_to_cover") * get_price(chain, debt_token, event.get("block_timestamp"), token_price)
        if protocols.get(contract_address) == "compound":
            underlying = underlyings.get(contract_address)
            amount = event.get("debt_to_cover") * exchange_rate.get(contract_address, 1) * get_price(chain, underlying, event.get("block_timestamp"), token_price)
        result[chain][str(round_time)]["events"] += 1
        result[chain][str(round_time)]["amount"] += amount
        if event.get("user") not in result[chain][str(round_time)]["wallets"]:
            result[chain][str(round_time)]["wallets"].append(event.get("user"))

In [42]:
import json
with open("analysis.json",'w') as f:
    json.dump(result, f, indent=1)

# Overview

In [1]:
import json
with open("analysis.json",'r') as f:
    data = json.loads(f.read())

In [9]:
import pandas
def descride_data(data, chain):
    chain_data = data[chain]
    data_dict = {'timestamp':[], 'wallets':[], 'events':[], 'amount':[]}
    wallets = []
    for timestamp, value in chain_data.items():
        wallets += value['wallets']
        data_dict['wallets'].append(len(value['wallets']))
        data_dict['events'].append(value['events'])
        data_dict['amount'].append(value['amount'])
        data_dict['timestamp'].append(timestamp)
    wallets = list(set(wallets))
    print(f'event/day: {sum(data_dict["events"])/len(data_dict["timestamp"])}')
    print(f'wallet/day: {sum(data_dict["wallets"])/len(data_dict["timestamp"])}')
    print(f'amount/day: {sum(data_dict["amount"])/len(data_dict["timestamp"])}')
    print(f'amount/event: {sum(data_dict["amount"])/sum(data_dict["events"])}')
    print(f'amount/wallet: {sum(data_dict["amount"])/len(wallets)}')
    print(f'event/wallet: {sum(data_dict["events"])/len(wallets)}')
    df = pandas.DataFrame(data_dict)
    return df

In [14]:
bnb_df = descride_data(data, "0x38")
bnb_df.describe()

event/day: 20.26086956521739
wallet/day: 16.22826086956522
amount/day: 23023.143590756474
amount/event: 1136.3354132776801
amount/wallet: 2038.6229166021133
event/wallet: 1.7940327237728586


,wallets,events,amount
count,92.000000,92.000000,9.200000e+01
mean,16.228261,20.260870,2.302314e+04
std,55.259051,77.983637,1.959709e+05
min,0.000000,0.000000,0.000000e+00
25%,1.000000,1.000000,1.589269e+00
50%,3.000000,3.000000,7.084126e+01
75%,8.250000,10.250000,6.870813e+02
max,437.000000,676.000000,1.878044e+06


In [15]:
poly_df = descride_data(data, "0x89")
poly_df.describe()

event/day: 118.67391304347827
wallet/day: 91.96739130434783
amount/day: 173106.7011499298
amount/event: 1458.6752615674611
amount/wallet: 3044.507074324898
event/wallet: 2.087172624737144


,wallets,events,amount
count,92.000000,92.000000,9.200000e+01
mean,91.967391,118.673913,1.731067e+05
std,323.410719,479.428027,1.014388e+06
min,4.000000,5.000000,1.133410e+01
25%,17.000000,18.000000,1.602433e+03
50%,26.000000,28.000000,4.372407e+03
75%,36.500000,43.000000,2.038414e+04
max,2802.000000,4343.000000,9.472857e+06


In [16]:
eth_df = descride_data(data, "0x1")
eth_df.describe()

event/day: 79.75
wallet/day: 61.130434782608695
amount/day: 3758930.8105696375
amount/event: 47133.928659180405
amount/wallet: 95267.66792628283
event/wallet: 2.021212121212121


,wallets,events,amount
count,92.000000,92.000000,9.200000e+01
mean,61.130435,79.750000,3.758931e+06
std,169.687251,277.534737,2.636876e+07
min,0.000000,0.000000,0.000000e+00
25%,3.000000,3.000000,1.321204e+03
50%,15.500000,16.000000,5.291620e+03
75%,50.000000,54.250000,6.309210e+04
max,1477.000000,2526.000000,2.508291e+08


In [17]:
arb_df = descride_data(data, "0xa4b1")
arb_df.describe()

event/day: 78.81521739130434
wallet/day: 64.6304347826087
amount/day: 362853.8267485423
amount/event: 4603.854924957371
amount/wallet: 7734.604277309058
event/wallet: 1.6800278035217795


,wallets,events,amount
count,92.000000,92.000000,9.200000e+01
mean,64.630435,78.815217,3.628538e+05
std,231.016517,327.294132,2.542485e+06
min,0.000000,0.000000,0.000000e+00
25%,3.000000,3.000000,5.976858e+01
50%,5.000000,5.000000,7.295275e+02
75%,13.000000,15.250000,7.613306e+03
max,1868.000000,2882.000000,2.411705e+07


# Adnormal

In [33]:
bnb_df[bnb_df["wallets"]>50].sort_values("wallets", ascending=False)

,timestamp,wallets,events,amount
35,1722816000,437,676,1.878044e+06
29,1722297600,216,216,6.375546e+01
4,1720137600,216,255,1.231665e+05
3,1720051200,75,87,2.485854e+04
34,1722729600,55,56,2.203034e+04


In [35]:
eth_df[eth_df["wallets"]>50].sort_values("wallets", ascending=False)

,timestamp,wallets,events,amount
35,1722816000,1477,2526,2.508291e+08
58,1724803200,436,660,5.381404e+04
34,1722729600,386,454,1.962455e+07
67,1725580800,333,385,1.859950e+07
4,1720137600,330,381,2.830358e+07
40,1723248000,174,193,7.030052e+03
62,1725148800,166,181,2.005439e+05
37,1722988800,135,154,1.919462e+05
59,1724889600,117,152,4.924343e+03
60,1724976000,115,135,8.806076e+04


In [34]:
poly_df[poly_df["wallets"]>50].sort_values("wallets", ascending=False)

,timestamp,wallets,events,amount
35,1722816000,2802,4343,9.472857e+06
4,1720137600,1198,1357,1.977706e+06
34,1722729600,680,794,1.371090e+06
3,1720051200,475,486,6.901275e+05
67,1725580800,282,302,3.523309e+05
57,1724716800,278,583,1.948748e+05
65,1725408000,180,190,3.064702e+05
68,1725667200,166,208,1.441004e+04
33,1722643200,136,141,7.414004e+04
69,1725753600,105,124,1.700812e+03


In [32]:
arb_df[arb_df["wallets"]>50].sort_values("wallets", ascending=False)

,timestamp,wallets,events,amount
35,1722816000,1868,2882,2.411705e+07
61,1725062400,706,744,4.482876e+01
37,1722988800,659,698,4.752905e+03
4,1720137600,629,700,3.712317e+06
34,1722729600,415,453,1.451058e+06
67,1725580800,288,299,1.355230e+06
66,1725494400,241,251,1.678871e+02
3,1720051200,221,245,6.948793e+05
65,1725408000,202,207,4.540470e+05
64,1725321600,60,64,5.306613e+01


# Normal Wallets

In [48]:
def get_normal_wallets(df, data, chain):
    timestamp = list(df["timestamp"][df["wallets"]>=200])
    wallets = []
    adnormal_wallets = []
    for key, value in data[chain].items():
        if key in timestamp:
            adnormal_wallets += value["wallets"]
        wallets += value["wallets"]

    wallets = list(set(wallets))
    adnormal_wallets = list(set(adnormal_wallets))
    normal_wallets = [w for w in wallets if w not in adnormal_wallets]
    print(f"total: {len(wallets)}")
    print(f"adnormal: {len(adnormal_wallets)}")
    print(f"normal: {len(normal_wallets)}")
    return normal_wallets, adnormal_wallets

In [49]:
bnb_nv, bnb_av = get_normal_wallets(bnb_df, data, "0x38")

total: 1039
adnormal: 763
normal: 276


In [50]:
poly_nv, poly_av = get_normal_wallets(poly_df, data, "0x89")

total: 5231
adnormal: 3764
normal: 1467


In [51]:
arb_nv, arb_av = get_normal_wallets(arb_df, data, "0xa4b1")

total: 4316
adnormal: 4005
normal: 311


In [52]:
eth_nv, eth_av = get_normal_wallets(eth_df, data, "0xa4b1")

total: 4316
adnormal: 2349
normal: 1967


In [53]:
normal_wallets = bnb_nv + poly_nv + arb_nv + eth_nv
adnormal_wallets = bnb_av + poly_av + arb_av + eth_av
normal_wallets = list(set(normal_wallets))
adnormal_wallets = list(set(adnormal_wallets))
print(len(normal_wallets))
print(len(adnormal_wallets))
normal_wallets = [i for i in normal_wallets if i not in adnormal_wallets]
print(len(normal_wallets))

3693
8297
2023


In [62]:
with open("debtor_type.json", "r") as f:
    debtor = json.loads(f.read())
result = []
for i in normal_wallets:
    if i in debtor and debtor[i] == 'contract':
        continue
    result.append(i)
print(len(result))

1918


In [65]:
no_data_wallets = []
for i in normal_wallets:
    if i not in debtor:
        no_data_wallets.append(i)
print(len(no_data_wallets))

357


In [80]:
chains = {
    "bnb": "0x38",
    "ethereum": "0x1",
    "polygon": "0x89",
    "arbitrum": "0xa4b1"
}


wallet_type = {}
cursor = klg_connection["knowledge_graph"]["wallets"].find({"address":{"$in": normal_wallets}}, ["type", "address", "tags"])
for wallet in cursor:
    address = wallet.get("address")
    if wallet.get(address) == "contract":
        continue
    if wallet.get("type") == "contract":
        wallet_type[address] = "contract"
    check = False
    for tag in wallet.get("tags", []):
        if "aave" in tag or "compound" in tag or "venus" in tag:
            check = True
            break
    if not check:
        continue
    if address not in wallet_type:
        wallet_type[address] = "wallet"
with open("debtor_type2.json", 'w') as f:
    json.dump(wallet_type, f, indent=1)

In [81]:
with open("debtor_type2.json", 'r') as f:
    wallet_type = json.loads(f.read())
wallet_debtors = [key for key, value in wallet_type.items() if value == "wallet"]
for idx in range(0, len(wallet_debtors), 1000):
    cursor = klg_connection["knowledge_graph"]["multichain_wallets"].find({"_id":{"$in": wallet_debtors[idx:idx+1000]}})
    for wallet in cursor:
        check = False
        for key in wallet.get("tokens", {}):
            chain_id = key.split("_")[0]
            if chain_id in ["0x1", "0xa4b1", "0x89", "0x38"]:
                check = True
                break
        if check:
            local_connection["knowledge_graph"]["multichain_wallets"].insert_one(wallet)

In [34]:
import pandas
from pymongo import MongoClient, UpdateOne

df = pandas.read_csv("data2/polygon_transaction.csv")
result = []
for idx, row in df.iterrows():
    data = {
  "_id": f"transaction_{row['hash']}",
  "block_hash": row["block_hash"],
  "block_number": row["block_number"],
  "block_timestamp": datetime.datetime.strptime(row["block_timestamp"], "%Y-%m-%d %H:%M:%S").timestamp(),
  "from_address": row["from_address"],
  "gas": f"{row['gas']}",
  "gas_price": str(row["gas_price"]),
  "hash": str(row["hash"]),
  "input": str(row["input"]),
  "nonce": row["nonce"],
  "receipt_contract_address": row["receipt_contract_address"],
  "receipt_cumulative_gas_used": str(row["receipt_cumulative_gas_used"]),
  "receipt_gas_used": str(row["receipt_gas_used"]),
  "receipt_status": row["receipt_status"],
  "to_address": str(row["to_address"]),
  "transaction_index": row["transaction_index"],
  "type": "transaction",
  "value": f"{row['value']}"
}
    update_data = UpdateOne({"_id": data["_id"]}, {"$set": data}, upsert=True)
    result.append(update_data)
local_database["poly_transactions"].bulk_write(result)

BulkWriteResult({'writeErrors': [], 'writeConcernErrors': [], 'nInserted': 0, 'nUpserted': 0, 'nMatched': 6197, 'nModified': 0, 'nRemoved': 0, 'upserted': []}, acknowledged=True)

In [51]:
df = pandas.read_csv("data2/bsc_transfers.csv")
decimals = {}
chain = '0x38'
result = []
for idx, row in df.iterrows():
    if row['contract_address'] not in decimals:
        token = klg_database["smart_contracts"].find_one({"_id":f"{chain}_{row['contract_address']}"})
        decimals[token.get('address')] = token.get("decimals", 18) or 18
    data = {
      "_id": f"{row['block_number']}_{row['log_index']}",
      "block_number": row['block_number'],
      "contract_address": row['contract_address'],
      "from_address": row['from_address'],
      "log_index": row['log_index'],
      "to_address": row['to_address'],
      "transaction_hash": row['transaction_hash'],
      "value": float(row['value'])/10**decimals.get(row['contract_address'])
}
    result.append(UpdateOne({"_id": data["_id"]}, {"$set": data}, upsert=True))
local_database["bnb_tf_events"].bulk_write(result)

BulkWriteResult({'writeErrors': [], 'writeConcernErrors': [], 'nInserted': 0, 'nUpserted': 29, 'nMatched': 0, 'nModified': 0, 'nRemoved': 0, 'upserted': [{'index': 0, '_id': '42760793_274'}, {'index': 1, '_id': '42832619_681'}, {'index': 2, '_id': '42629248_112'}, {'index': 3, '_id': '42629272_176'}, {'index': 4, '_id': '42629317_827'}, {'index': 5, '_id': '42629926_115'}, {'index': 6, '_id': '42629926_134'}, {'index': 7, '_id': '42693378_37'}, {'index': 8, '_id': '42694926_99'}, {'index': 9, '_id': '42694978_371'}, {'index': 10, '_id': '42695056_301'}, {'index': 11, '_id': '42695056_304'}, {'index': 12, '_id': '39857496_144'}, {'index': 13, '_id': '39857502_176'}, {'index': 14, '_id': '43147196_15'}, {'index': 15, '_id': '43147220_98'}, {'index': 16, '_id': '43147283_152'}, {'index': 17, '_id': '43147341_155'}, {'index': 18, '_id': '43135177_1041'}, {'index': 19, '_id': '42731767_148'}, {'index': 20, '_id': '42732137_61'}, {'index': 21, '_id': '42732141_57'}, {'index': 22, '_id': '427

In [2]:
import json
with open("multichain_wallets.json", 'r') as f:
    data = json.loads(f.read())
addresses = [i['address'] for i in data]
foundation = []
for i in klg_database['wallets'].find({"address":{"$in": addresses}}, projection=["foundation", "address"]):
    if i.get('foundation'):
        foundation.append(i.get('address'))

In [5]:
with open("multichain_wallets.json", 'w') as f:
    json.dump(addresses, f, indent=1)